# RA-XAEGE A+C — Retrieval-Aware Explainable KV Cache
### Combine Opsi B (Pisah Tema) + Opsi C (XAI) + Opsi A (Retrieval-Guided) untuk Profile Alignment Historically Cocok

**Why old results jelek (85.7% vs Streaming 92.9%):**
- Window tidak fair: `Streaming window = budget-4 = 252` vs AEGE `64` → streaming dapat konteks 4x gratis
- Task trivial: output cuma `Score: X` 5 token langsung EOS, eviction di decode tidak ngaruh
- Entropy bug: elementwise `-(p log p)` bukan Shannon `H = -sum p log p`
- Adaptive mati: CSV `256:256:...` uniform, peak 7168 bukan 6440 pyramid

**Fix A+C 4 Level (PhD level):**
1. Rubric-Aware Protection (B) — never evict rubric + student essay spans
2. Retrieval-Guided Importance (A) — `score = attn * (1-entropy) * (1+retrieval_weight)` — consume S2 RAG relevance 0.92 vs 0.3
3. Entropy XAI Fixed (C) — proper Shannon per key + clamp min 0.05 + quantile 30%
4. Pyramid Adaptive — shallow 60% deep 120% per layer

**Profile Alignment Historically:** S2 = XAI for AES (SHAP important sentences), S3 = XAI for Cache (keep important tokens). Identity = Explainable AI Researcher, bukan cuma AES.
**New Task:** Feedback 150 kata + Score, budget sweep [1024,512,256,128] → 3 kurva: fidelity, efficiency, XAI IoU

In [ ]:
# ==========================================
# Cell 0: Clone / Update Repo & Install (FIXED absolute path)
# From commit 9636b51: use /content/kv-cache-testing hard reset
# ==========================================
import os
%cd /content
if not os.path.exists('/content/kv-cache-testing'):
    !git clone https://github.com/danielwidhiarto/kv-cache-testing.git /content/kv-cache-testing
%cd /content/kv-cache-testing
!git fetch origin && git reset --hard origin/main && git pull origin main

!pip install -q torch transformers accelerate pandas matplotlib sentencepiece protobuf scikit-learn
print("✅ Repo ready at /content/kv-cache-testing")

In [ ]:
# ==========================================
# Cell 1: Quick smoke test RA-XAEGE (no GPU needed, gpt2 logic)
# ==========================================
import torch
from src.policies.aege import AEGEPolicy
from src.policies.ra_aege import RAXAEGEPolicy
from src.metrics.xai_metrics import xai_fidelity_iou, rubric_retention_rate, attention_importance_topk

p = AEGEPolicy(sink_size=4, window_size=64, entropy_weight=0.5, adaptive_budget=True, adaptive_quantile=0.3)
p.layer_idx=0; p.total_layers=28
print("shallow budget", p.get_max_size(256))
p.layer_idx=27
print("deep budget", p.get_max_size(256))

attn = torch.rand(1,2,10,20)
ent = p._compute_entropy_per_key(attn)
print("entropy shape", ent.shape, "min", ent.min().item(), "max", ent.max().item())

k=torch.rand(1,2,25,32)
v=torch.rand(1,2,25,32)
ra=RAXAEGEPolicy(window_size=64, entropy_weight=0.5, retrieval_spans=[{"start":10,"end":15,"score":0.9},{"start":16,"end":18,"score":0.3}], rubric_spans=[(0,5)], never_evict_spans=[(20,25)])
idx=ra.select_evict(k,v, attention_scores=torch.rand(1,2,25,25), num_to_evict=5)
print("RA-XAEGE evict indices (should avoid 0-5 rubric, 20-25 student, keep 10-15 high relevance):", idx.tolist())

# XAI metrics
kept = list(range(0,25))
kept = [x for x in kept if x not in idx.tolist()]
imp = attention_importance_topk(torch.rand(25), 0.3)
print("XAI IoU mock", xai_fidelity_iou(kept, imp.tolist()))
print("Rubric retention", rubric_retention_rate(kept, [(0,5)]))

In [ ]:
# ==========================================
# Cell 2: RAG Prompt Example + char→token span mapping
# Shows how S2 output consumed by S3
# ==========================================
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

prompt_example = """You are expert AES evaluator.
### SOURCE READING TEXT
Venus is second planet...

### GRADING RUBRIC
Score 6: comprehensive understanding... Score 1: minimal

### RETRIEVED EXEMPLARS (k=3 from S2 multi-retriever)
[Exemplar 1 | Relevance=0.92 | HumanScore=5] Essay: Venus presents extraordinary challenge...
[Exemplar 2 | Relevance=0.45 | HumanScore=2] Essay: Venus is planet...
[Exemplar 3 | Relevance=0.32 | HumanScore=1] Essay: I think Venus...

### STUDENT ESSAY TO EVALUATE
This essay discusses Venus exploration challenges...

### TASK
Provide detailed feedback (120-150 words) + Score: X
"""

enc = tokenizer(prompt_example, return_offsets_mapping=True)
offsets = enc["offset_mapping"]
print(f"Tokens: {len(enc['input_ids'])} chars: {len(prompt_example)}")

# simulate char spans from src/utils/aes_loader + RAG
char_spans = {"rubric": [(50, 120)], "retrieved": [{"char_start": 130, "char_end": 200, "score":0.92}, {"char_start": 200, "char_end": 250, "score":0.45}], "student": [(300, 400)]}

def char_to_token(char_s, char_e):
    res=[]
    for i,(os_s, os_e) in enumerate(offsets):
        if os_e <= char_s: continue
        if os_s >= char_e: break
        res.append(i)
    return res

for span in char_spans["retrieved"]:
    toks = char_to_token(span["char_start"], span["char_end"])
    print(f"Retrieved relevance {span['score']} -> tokens {toks[:5]}... count {len(toks)} -> should NOT evict high relevance 0.92")

print("✅ Char→token mapping works for A+C")

In [ ]:
# ==========================================
# Cell 3: Main Benchmark — RA-XAEGE A+C
# Budget sweep PhD ready
# ==========================================
# Fair window 64 all policies (fix streaming 252 cheat)
# Task: Feedback 150 tok + Score (not 5 tok Score)
!python benchmarks/bench_aes_v2_raxaege.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --num-samples 14 \
    --max-cache-sizes 1024 512 256 128 \
    --max-new-tokens 150 \
    --retrieval-k 3 \
    --policies full aege aege_adaptive ra_xaege h2o streaming lru \
    --output-dir results

print("Quick test single budget if sweep too heavy:")
# Uncomment if only 256 budget needed
# !python benchmarks/bench_aes_v2_raxaege.py --model Qwen/Qwen2.5-7B-Instruct --num-samples 14 --max-cache-size 256 --max-new-tokens 150 --retrieval-k 3 --policies full aege ra_xaege streaming lru --output-dir results

In [ ]:
# ==========================================
# Cell 4: Analysis + PhD curves
# ==========================================
import pandas as pd, os
import matplotlib.pyplot as plt

csv_path = "results/aes_benchmark_v2_raxaege.csv"
if not os.path.exists(csv_path):
    csv_path = "results/aes_benchmark.csv"  # fallback old
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows, policies {df['policy'].unique().tolist()}")

budget_col = "budget" if "budget" in df.columns else "max_cache_size"
if "score_match_vs_fullcache" in df.columns:
    agg = df.groupby([budget_col, "policy"]).agg(match=("score_match_vs_fullcache","mean"), tput=("throughput_tok_sec","mean"), peak=("peak_cache_tokens","mean"), latency=("latency_sec","mean")).reset_index()
    agg["match_pct"] = agg["match"]*100
    display(agg)
    
    # Plot 1: Match vs Budget
    plt.figure(figsize=(7,4.5))
    for pol in agg["policy"].unique():
        if pol=="FullCache": continue
        sub=agg[agg["policy"]==pol].sort_values(budget_col)
        plt.plot(sub[budget_col], sub["match_pct"], marker="o", label=pol)
    plt.axhline(100, ls="--", color="gray")
    plt.xlabel("Budget per layer")
    plt.ylabel("Score Match vs FullCache (%)")
    plt.title("RA-XAEGE A+C: Fidelity vs Budget (Fair Window 64, Feedback 150tok)")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.savefig("results/figures_sweep_match.png", dpi=200)
    plt.show()
    
    # Plot 2: Throughput
    plt.figure(figsize=(7,4.5))
    for pol in agg["policy"].unique():
        sub=agg[agg["policy"]==pol].sort_values(budget_col)
        plt.plot(sub[budget_col], sub["tput"], marker="s", label=pol)
    plt.xlabel("Budget")
    plt.ylabel("Throughput tok/s")
    plt.title("Throughput vs Budget")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.savefig("results/figures_sweep_throughput.png", dpi=200)
    plt.show()
else:
    print("Old CSV detected without match column — plotting latency/throughput only")
    display(df.groupby("policy")[["latency_sec","throughput_tok_sec","peak_cache_tokens"]].mean())

In [ ]:
# ==========================================
# Cell 5: XAI IoU — S2 SHAP vs S3 Kept tokens (Profile Alignment)
# ==========================================
import torch
from src.metrics.xai_metrics import xai_fidelity_iou, attention_importance_topk

# Mock: FullCache attention importance as proxy for S2 SHAP ground truth
# Real: replace with json.load open("s2_shap_scores.json") -> important token indices
# Example: S2 SHAP says tokens [10,11,12,45,46,100-110] important for Score 4

mock_shap_important = list(range(10,20)) + list(range(100,115)) + list(range(200,210))
mock_kept_aege = list(range(0,5)) + list(range(10,20)) + list(range(95,115)) + list(range(200,220)) + list(range(400,464))
mock_kept_streaming = list(range(0,4)) + list(range(400,464))  # only sink+window, no middle important
mock_kept_ra_xaege = mock_kept_aege + list(range(80,90))  # retrieval guided keeps more

print(f"S2 SHAP important count: {len(mock_shap_important)}")
print(f"AEGE IoU/recall: {xai_fidelity_iou(mock_kept_aege, mock_shap_important):.3f}")
print(f"Streaming IoU/recall: {xai_fidelity_iou(mock_kept_streaming, mock_shap_important):.3f}")
print(f"RA-XAEGE IoU/recall: {xai_fidelity_iou(mock_kept_ra_xaege, mock_shap_important):.3f}")
print("\nInterpretation: RA-XAEGE keeps more S2-important tokens → explainable cache. This is B+C profile alignment (XAI researcher).")

# Visual highlight
def visualize_keep(text_tokens, kept_set, important_set):
    html="<div style='font-family:monospace;font-size:12px;line-height:1.6'>"
    for i, tok in enumerate(text_tokens[:80]):
        color="#fff"
        if i in important_set and i in kept_set:
            color="#86efac"  # green both
        elif i in important_set and i not in kept_set:
            color="#fca5a5"  # red important but evicted
        elif i in kept_set:
            color="#bfdbfe"  # blue kept
        html+=f"<span style='background:{color};padding:2px;margin:1px'>{tok}</span> "
    html+="</div><p style='font-size:11px'>Green=important+kept, Red=important+evicted, Blue=kept only</p>"
    return html

from IPython.display import HTML
tokens_mock = [f"tok{i}" for i in range(80)]
HTML(visualize_keep(tokens_mock, set(mock_kept_ra_xaege), set(mock_shap_important)))

In [ ]:
# ==========================================
# Cell 6: Ablation A+C
# ==========================================
print("Ablation design for thesis chapter:")
print("- ra_xaege full: rubric protection + retrieval prior (rw=0.7) + entropy 0.5 + pyramid")
print("- aege_adaptive: entropy only, no retrieval, rubric yes")
print("- aege: fixed, no retrieval, no rubric? actually rubric yes now")
print("- h2o: heavy hitter only, no entropy, no rubric")
print("- streaming: sink+window only, no attention")
print("- lru: recency only")
print("\nRun ablation quickly for 256 budget:")
!python benchmarks/bench_aes_v2_raxaege.py --model Qwen/Qwen2.5-7B-Instruct --num-samples 8 --max-cache-size 256 --max-new-tokens 150 --retrieval-k 3 --policies full aege aege_adaptive ra_xaege h2o streaming lru --output-dir results/ablation

In [ ]:
# ==========================================
# Cell 7: Generate Dashboard v2 + Plots
# ==========================================
!python benchmarks/plot_sweep.py --input results/aes_benchmark_v2_raxaege.csv --output-dir results/figures

import pandas as pd, os
df=pd.read_csv("results/aes_benchmark_v2_raxaege.csv") if os.path.exists("results/aes_benchmark_v2_raxaege.csv") else pd.read_csv("results/aes_benchmark.csv")
print(df.head())

# Simple HTML dashboard v2 with curves
html = f"""
<html><head><title>RA-XAEGE A+C Dashboard</title></head><body style='font-family: sans-serif'>
<h1>RA-XAEGE A+C — Budget Sweep Results</h1>
<p>Policies: {df['policy'].unique().tolist()} | Rows: {len(df)}</p>
<img src='figures/sweep_match_vs_budget.png' width='700'/><br/>
<img src='figures/sweep_throughput.png' width='700'/><br/>
<img src='figures/sweep_peak.png' width='700'/>
</body></html>
"""
open("results/dashboard_v2.html","w").write(html)
print("Saved results/dashboard_v2.html")

In [ ]:
# ==========================================
# Cell 8: Download results (Colab copy github flow)
# ==========================================
from google.colab import files
import os
for f in ["results/aes_benchmark_v2_raxaege.csv", "results/dashboard_v2.html", "results/figures/sweep_match_vs_budget.png"]:
    if os.path.exists(f):
        print(f"Downloading {f}")
        files.download(f)
    else:
        print(f"Skip {f} not found")